# Media Analytics – ETL Pipeline
### Traditional vs. Digital Media Consumption Shift

**Target Schema:** Constellation — 3 fact tables, 5 non-time dimensions, 1 time dimension  
**Fact Tables:** `FACT_MEDIA_PERFORMANCE` (Cumulative), `FACT_ENGAGEMENT` (Snapshot), `FACT_SUBSCRIPTION_PRICING` (Snapshot)  
**SCD:** DIM_PLATFORM → SCD2 | DIM_SUBSCRIPTION_PLAN → SCD3 | others → SCD0/SCD1

---
**Notebook Structure**
- Part 1: Load & Profile All Data Sources
- Part 2: Transformations — streaming_service.csv
- Part 3: Transformations — platform_summary.csv
- Part 4: DIM_PLATFORM Initial Load (SCD2)

In [ ]:
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings('ignore')

pd.set_option('display.max_columns', 20)
pd.set_option('display.width', 120)

---
## Part 1 – Load & Profile All Data Sources

| # | File | Rows | Feeds |
|---|---|---|---|
| 1 | `streaming_service.csv` | 777 | FACT_SUBSCRIPTION_PRICING, DIM_SUBSCRIPTION_PLAN |
| 2 | `platform_summary.csv` | 12 | DIM_PLATFORM |
| 3 | `platform_financials_comprehensive.csv` | 10 | FACT_MEDIA_PERFORMANCE |
| 4 | `industry_metrics.csv` | 9 | FACT_MEDIA_PERFORMANCE |
| 5 | `traditional_media_viewership_monthly.csv` | 1,624 | FACT_MEDIA_PERFORMANCE |
| 6 | `user_engagement_monthly.csv` | 4,120 | FACT_ENGAGEMENT |
| 7 | `switch_factor_survey.csv` | 2,025 | FACT_ENGAGEMENT, DIM_SWITCH_REASON |
| 8 | `platform_subscriber_monthly.csv` | 640 | FACT_MEDIA_PERFORMANCE |

### 1.1 – streaming_service.csv
**Source:** Kaggle | **Feeds:** FACT_SUBSCRIPTION_PRICING, DIM_SUBSCRIPTION_PLAN  
**Known issues:** Date stored as `Jul-2011` string format — needs parsing

In [ ]:
StreamingService_df = pd.read_csv('Data Sources copy/streaming_service.csv')

print('Shape:', StreamingService_df.shape)
print('Columns:', list(StreamingService_df.columns))
print('\nData types:')
print(StreamingService_df.dtypes)
print('\nNull counts:')
print(StreamingService_df.isnull().sum())
print('\nSample rows:')
display(StreamingService_df.head(5))
print('\nUnique services:', StreamingService_df['service'].nunique())
print('Date range (raw):', StreamingService_df['date'].min(), '→', StreamingService_df['date'].max())

### 1.2 – platform_summary.csv
**Source:** Streaming Platform Dataset | **Feeds:** DIM_PLATFORM  
**Known issues:** Wide table (36 cols), many NaN — only 5 dim cols needed

In [ ]:
PlatformSummary_df = pd.read_csv('Data Sources copy/Streaming Platform Dataset/platform_summary.csv')

print('Shape:', PlatformSummary_df.shape)
print('\nDimension columns (selected for DIM_PLATFORM):')
dim_cols = ['platform', 'parent_company', 'content_focus', 'business_model', 'launch_year']
print(PlatformSummary_df[dim_cols].to_string())
print('\nNull counts (dim cols only):')
print(PlatformSummary_df[dim_cols].isnull().sum())

### 1.3 – platform_financials_comprehensive.csv
**Source:** Streaming Platform Dataset | **Feeds:** FACT_MEDIA_PERFORMANCE  
**Known issues:** 10 rows only (1 per platform, snapshot); many sparse columns

In [ ]:
PlatformFinancials_df = pd.read_csv('Data Sources copy/Streaming Platform Dataset/platform_financials_comprehensive.csv')

print('Shape:', PlatformFinancials_df.shape)
print('Platforms:', list(PlatformFinancials_df['platform']))

# Key financial columns relevant to FACT_MEDIA_PERFORMANCE
fin_cols = ['platform', 'reporting_date', 'subscribers_millions',
            'quarterly_revenue_usd_millions', 'quarterly_profit_usd_millions']
print('\nKey columns:')
display(PlatformFinancials_df[fin_cols])
print('\nNull counts (key cols):')
print(PlatformFinancials_df[fin_cols].isnull().sum())

### 1.4 – industry_metrics.csv
**Source:** Streaming Platform Dataset | **Feeds:** FACT_MEDIA_PERFORMANCE  
**Known issues:** Annual granularity only; industry-level aggregates, not platform-level

In [ ]:
IndustryMetrics_df = pd.read_csv('Data Sources copy/Streaming Platform Dataset/industry_metrics.csv')

print('Shape:', IndustryMetrics_df.shape)
print('\nAll rows:')
display(IndustryMetrics_df)
print('\nNull counts:')
print(IndustryMetrics_df.isnull().sum())

### 1.5 – traditional_media_viewership_monthly.csv
**Source:** Synthetic (AI-generated) | **Feeds:** FACT_MEDIA_PERFORMANCE  
**Known issues:** Mixed date formats (`Jan-2010`, `2010/01`, `March 2010`), some `metric_value` stored as strings (`"13.6M"`), inconsistent `media_type` casing, 25 intentional duplicate rows

In [ ]:
TraditionalMedia_df = pd.read_csv('Data Sources copy/traditional_media_viewership_monthly.csv')

print('Shape:', TraditionalMedia_df.shape)
print('Columns:', list(TraditionalMedia_df.columns))
print('\nData types:')
print(TraditionalMedia_df.dtypes)
print('\nNull counts:')
print(TraditionalMedia_df.isnull().sum())
print('\nSample rows:')
display(TraditionalMedia_df.head(8))

print('\n--- Data Quality Issues ---')
print('Duplicate rows:', TraditionalMedia_df.duplicated().sum())
print('\nUnique date formats (sample):', TraditionalMedia_df['report_month'].unique()[:10])
print('\nmedia_type unique values:', TraditionalMedia_df['media_type'].unique())
print('\nmetric_value stored as string (sample):')
str_vals = TraditionalMedia_df[TraditionalMedia_df['metric_value'].astype(str).str.contains('M', na=False)]
print(str_vals[['platform_name','report_month','metric_value']].head(5))

### 1.6 – user_engagement_monthly.csv
**Source:** Synthetic (AI-generated) | **Feeds:** FACT_ENGAGEMENT  
**Known issues:** Mixed date formats (`2015-01` vs `01/2015`), `retention_rate_pct` stored as decimal in some rows and `"78%"` string in others, nulls for new platforms in early months, 40 intentional duplicate rows

In [ ]:
UserEngagement_df = pd.read_csv('Data Sources copy/user_engagement_monthly.csv')

print('Shape:', UserEngagement_df.shape)
print('Columns:', list(UserEngagement_df.columns))
print('\nData types:')
print(UserEngagement_df.dtypes)
print('\nNull counts:')
print(UserEngagement_df.isnull().sum())
print('\nSample rows:')
display(UserEngagement_df.head(6))

print('\n--- Data Quality Issues ---')
print('Duplicate rows:', UserEngagement_df.duplicated().sum())
print('\nMixed date formats (sample):', UserEngagement_df['year_month'].unique()[:10])
print('\nretention_rate_pct mixed types (sample):')
print(UserEngagement_df['retention_rate_pct'].head(20).tolist())

### 1.7 – switch_factor_survey.csv
**Source:** Synthetic (AI-generated) | **Feeds:** FACT_ENGAGEMENT, DIM_SWITCH_REASON  
**Known issues:** Inconsistent `switched_from` values (`Cable` vs `Cable TV` vs `cable tv`), many null `secondary_switch_reason`, duplicate `respondent_id` across survey years (re-contact design), mixed `satisfaction_score` types

In [ ]:
SwitchSurvey_df = pd.read_csv('Data Sources copy/switch_factor_survey.csv')

print('Shape:', SwitchSurvey_df.shape)
print('Columns:', list(SwitchSurvey_df.columns))
print('\nData types:')
print(SwitchSurvey_df.dtypes)
print('\nNull counts:')
print(SwitchSurvey_df.isnull().sum())
print('\nSample rows:')
display(SwitchSurvey_df.head(5))

print('\n--- Data Quality Issues ---')
print('switched_from unique values:', SwitchSurvey_df['switched_from'].unique())
print('\nDuplicate respondent_ids (re-contact across years):')
print(SwitchSurvey_df[SwitchSurvey_df.duplicated('respondent_id', keep=False)][['survey_year','respondent_id']].head(6).to_string())
print('\nRows per survey year:')
print(SwitchSurvey_df.groupby('survey_year').size())

### 1.8 – platform_subscriber_monthly.csv
**Source:** Synthetic (AI-generated, anchored to real quarterly figures) | **Feeds:** FACT_MEDIA_PERFORMANCE  
**Known issues:** Null `revenue_usd_millions` for some platforms; seasonal churn spikes built in

In [ ]:
PlatformSubscriber_df = pd.read_csv('Data Sources copy/platform_subscriber_monthly.csv')

print('Shape:', PlatformSubscriber_df.shape)
print('Columns:', list(PlatformSubscriber_df.columns))
print('\nData types:')
print(PlatformSubscriber_df.dtypes)
print('\nNull counts:')
print(PlatformSubscriber_df.isnull().sum())
print('\nSample rows:')
display(PlatformSubscriber_df.head(5))
print('\nPlatforms covered:', PlatformSubscriber_df['platform_name'].unique())
print('Date range:', PlatformSubscriber_df['year_month'].min(), '→', PlatformSubscriber_df['year_month'].max())

---
## Part 2 – Transformations: streaming_service.csv
**Target:** DIM_SUBSCRIPTION_PLAN (SCD3) and FACT_SUBSCRIPTION_PRICING

#### T1 – Remove Duplicates

In [ ]:
StreamingServiceDistinct_df = StreamingService_df.drop_duplicates()

print('Rows before:', StreamingService_df.shape[0])
print('Rows after dedup:', StreamingServiceDistinct_df.shape[0])
print('Records removed:', StreamingService_df.shape[0] - StreamingServiceDistinct_df.shape[0])

print('\nNull counts after dedup:')
print(StreamingServiceDistinct_df.isnull().sum())

#### T2 – Date Standardization
Parse `date` from `%b-%Y` string format (e.g. `Jul-2011`) to proper `datetime`

In [ ]:
print('Before:')
print(StreamingServiceDistinct_df[['service', 'date', 'price']].head(5).to_string())

StreamingServiceDistinct_df['date'] = pd.to_datetime(
    StreamingServiceDistinct_df['date'], format='%b-%Y'
)

print('\nAfter:')
print(StreamingServiceDistinct_df[['service', 'date', 'price']].head(5).to_string())

#### T3 – Price Tier Classification
Derive `price_tier`: Budget ($0–$7) | Mid ($7–$14) | Premium (>$14)

In [ ]:
print('Before — price distribution:')
print(StreamingServiceDistinct_df['price'].describe())

bins   = [0, 7, 14, float('inf')]
labels = ['Budget ($0-$7)', 'Mid ($7-$14)', 'Premium (>$14)']
StreamingServiceDistinct_df['price_tier'] = pd.cut(
    StreamingServiceDistinct_df['price'], bins=bins, labels=labels, right=True
)

print('\nAfter — sample rows:')
print(StreamingServiceDistinct_df[['service', 'date', 'price', 'price_tier']].head(10).to_string())
print('\nTier distribution:')
print(StreamingServiceDistinct_df['price_tier'].value_counts())

#### T4 – Month-over-Month Price Change
Compute `price_change_mom` = diff of `price` per service sorted by date

In [ ]:
StreamingServiceDistinct_df = StreamingServiceDistinct_df.sort_values(['service', 'date'])

StreamingServiceDistinct_df['price_change_mom'] = (
    StreamingServiceDistinct_df.groupby('service')['price'].diff()
)

print('Months where price changed:')
price_changes = (
    StreamingServiceDistinct_df[
        (StreamingServiceDistinct_df['price_change_mom'] != 0) &
        StreamingServiceDistinct_df['price_change_mom'].notna()
    ]
)
print(price_changes[['service', 'date', 'price', 'price_change_mom']].to_string())

#### T5 – Cumulative Price Increase
Compute `cumulative_price_increase` = current price minus the service's first recorded price

In [ ]:
StreamingServiceDistinct_df['cumulative_price_increase'] = (
    StreamingServiceDistinct_df.groupby('service')['price']
    .transform(lambda x: x - x.iloc[0])
)

print('Cumulative price increase per service (latest month):')
latest = (
    StreamingServiceDistinct_df
    .sort_values('date')
    .groupby('service')
    .last()
    .reset_index()
)
print(latest[['service', 'date', 'price', 'cumulative_price_increase']].to_string())

#### T6 – Service Name Normalization
Map legacy service names to canonical platform names for cross-source joins

In [ ]:
name_map = {
    'HBO Max'     : 'Max',
    'Prime Video' : 'Amazon Prime Video'
}
StreamingServiceDistinct_df['platform_name'] = StreamingServiceDistinct_df['service'].replace(name_map)

print('Service → platform_name mapping:')
print(StreamingServiceDistinct_df[['service', 'platform_name']].drop_duplicates().to_string())

---
## Part 3 – Transformations: platform_summary.csv
**Target:** DIM_PLATFORM

#### T7 – Select Dimension Columns + Remove Duplicates

In [ ]:
dim_cols = ['platform', 'parent_company', 'content_focus', 'business_model', 'launch_year']
PlatformSummaryDistinct_df = PlatformSummary_df[dim_cols].drop_duplicates()

print('Rows before:', PlatformSummary_df.shape[0])
print('Rows after dedup:', PlatformSummaryDistinct_df.shape[0])
print('\nNull counts (dim cols):')
print(PlatformSummaryDistinct_df.isnull().sum())
print('\nAll rows:')
display(PlatformSummaryDistinct_df)

#### T8 – Derived Columns for DIM_PLATFORM
Add `is_digital`, `media_sector`, `platform_age_years`, `launch_decade`

In [ ]:
print('Before:')
display(PlatformSummaryDistinct_df[['platform', 'launch_year']])

PlatformSummaryDistinct_df = PlatformSummaryDistinct_df.copy()
PlatformSummaryDistinct_df['is_digital']         = True
PlatformSummaryDistinct_df['media_sector']        = 'Streaming'
PlatformSummaryDistinct_df['platform_age_years']  = 2026 - PlatformSummaryDistinct_df['launch_year'].astype(int)
PlatformSummaryDistinct_df['launch_decade']       = (
    (PlatformSummaryDistinct_df['launch_year'].astype(int) // 10 * 10).astype(str) + 's'
)

print('\nAfter:')
display(PlatformSummaryDistinct_df[['platform', 'launch_year', 'launch_decade', 'is_digital', 'media_sector', 'platform_age_years']])

#### T9 – Cross-Source Platform Alignment
Identify platforms in pricing data that are missing from platform_summary (need manual enrichment)

In [ ]:
services_in_pricing  = set(StreamingServiceDistinct_df['platform_name'].unique())
platforms_in_summary = set(PlatformSummaryDistinct_df['platform'].unique())

print('In pricing but NOT in platform_summary (will add manually):')
print(sorted(services_in_pricing - platforms_in_summary))

print('\nIn platform_summary but NOT in pricing (no pricing history):')
print(sorted(platforms_in_summary - services_in_pricing))

---
## Part 4 – DIM_PLATFORM: Initial Load (SCD2)
Build the platform dimension with SCD2 metadata columns.  
SCD2-tracked attributes: `parent_company`, `business_model`, `media_sector`

#### Step 1 – Add missing platforms (manual enrichment)

In [ ]:
# Platforms in pricing data not covered by platform_summary — built from metadata
NewPlatformRows_df = pd.DataFrame([
    {
        'platform': 'Crunchyroll', 'parent_company': 'Sony Group Corporation',
        'content_focus': 'Anime, manga', 'business_model': 'Subscription',
        'launch_year': 2009, 'is_digital': True, 'media_sector': 'Streaming',
        'platform_age_years': 2026 - 2009, 'launch_decade': '2000s'
    },
    {
        'platform': 'Shudder', 'parent_company': 'AMC Networks',
        'content_focus': 'Horror, thriller', 'business_model': 'Subscription',
        'launch_year': 2015, 'is_digital': True, 'media_sector': 'Streaming',
        'platform_age_years': 2026 - 2015, 'launch_decade': '2010s'
    },
])

print('New platform rows added:')
display(NewPlatformRows_df)

#### Step 2 – Concat + SCD2 metadata columns

In [ ]:
DimPlatform = pd.concat([PlatformSummaryDistinct_df, NewPlatformRows_df], ignore_index=True)

# SCD2 metadata — initial load: all rows are current with no end date
DimPlatform['effective_date'] = pd.to_datetime(DimPlatform['launch_year'].astype(str) + '-01-01')
DimPlatform['end_date']       = None
DimPlatform['is_current']     = True

# Surrogate key
DimPlatform['platform_key'] = range(1, len(DimPlatform) + 1)

print('Rows in DimPlatform:', len(DimPlatform))
print('\nFinal DIM_PLATFORM:')
display(DimPlatform[[
    'platform_key', 'platform', 'parent_company', 'media_sector',
    'business_model', 'launch_year', 'launch_decade', 'is_digital',
    'platform_age_years', 'effective_date', 'end_date', 'is_current'
]])

#### Step 3 – SCD2 Upsert Demo: HBO Max → Max rebrand (May 2023)

Simulates an incoming delta where `platform_name` and `content_focus` changed.  
Process: expire old row → insert new row with new surrogate key.

In [ ]:
scd2_change_date = pd.Timestamp('2023-05-23')

# --- Before ---
print('=== BEFORE SCD2 upsert ===')
display(DimPlatform[DimPlatform['platform'] == 'Max'][[
    'platform_key', 'platform', 'parent_company', 'business_model',
    'effective_date', 'end_date', 'is_current'
]])

# Step 1: expire the existing row for Max (formerly HBO Max)
mask = (DimPlatform['platform'] == 'Max') & (DimPlatform['is_current'] == True)
DimPlatform.loc[mask, 'end_date']   = scd2_change_date
DimPlatform.loc[mask, 'is_current'] = False

# Step 2: insert new row with updated values and new surrogate key
old_row = DimPlatform[DimPlatform['platform'] == 'Max'].iloc[0].to_dict()
new_row = old_row.copy()
new_row['platform_key']   = DimPlatform['platform_key'].max() + 1
new_row['platform']       = 'Max'
new_row['business_model'] = 'Subscription + Ads'   # changed
new_row['effective_date'] = scd2_change_date
new_row['end_date']       = None
new_row['is_current']     = True

DimPlatform = pd.concat([DimPlatform, pd.DataFrame([new_row])], ignore_index=True)

# --- After ---
print('\n=== AFTER SCD2 upsert — both rows preserved ===')
display(DimPlatform[DimPlatform['platform'] == 'Max'][[
    'platform_key', 'platform', 'parent_company', 'business_model',
    'effective_date', 'end_date', 'is_current'
]])